# M08-1 | Two XNNPACKQuantizers with the Same Name: the executorch Version vs the torchao Test Copy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobondemon/pt2e_quantize_and_deploy/blob/main/en/m08_xnnpack_executorch/01_executorch_vs_torchao_copy.ipynb)

PT2E has two `XNNPACKQuantizer`s with the same name. The code is almost identical, but the behavior differs:

| Name | import path | What it is |
|---|---|---|
| **executorch version** | `executorch.backends.xnnpack.quantizer.xnnpack_quantizer` | The quantizer shipped with ExecuTorch's XNNPACK backend, in the same package as `XnnpackPartitioner`; use this one to export a `.pte` |
| **torchao test copy** | `torchao.testing.pt2e._xnnpack_quantizer` | A copy torchao keeps for its own unit tests (under `testing`, with a leading `_`; importing it emits a deprecation warning) |

M01 to M07 all used the torchao test copy, so results do not change depending on whether executorch is installed; from M08 on we switch to the executorch version. Before switching, check how the two differ:

- **A.** On the same CatPoolNet, which ops each one annotates.
- **B.** The executorch version's quantization specs: static (default) and dynamic modes.
- **C.** Whether the module-level setting `set_module_name` takes effect on conv (continuing M03-7).

This notebook needs executorch (`uv sync --group executorch`).

In [ ]:
#| include: false
# On Google Colab: download this tutorial's code and install the packages (takes a few minutes the first time).
# On your own machine (see the README for the environment), this cell does nothing.
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    repo = "/content/pt2e_quantize_and_deploy"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/bobondemon/pt2e_quantize_and_deploy.git", repo], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--extra-index-url", "https://download.pytorch.org/whl/cpu",
                        "torch==2.14.0+cpu", "torchao==0.18.0", "onnx==1.23.0", "onnxscript==0.7.2",
                        "onnxruntime==1.30.0", "executorch==1.5.0", "tabulate", "graphviz"], check=True)
    sys.path.insert(0, repo)  # let the notebook import this tutorial's common/
    os.chdir(f"{repo}/en/m08_xnnpack_executorch")

In [ ]:
import torch
import torch.nn as nn

from common.graph_utils import diff_ops, qspec_summary, short
from common.models import CatPoolNet, TinyConvNet, example_inputs
from common.pt2e import ANNOTATION_KEY, convert_pt2e, export_module, get_xnnpack_api, get_xnnpack_quantizer, prepare_pt2e, quiet
from common.viz import annotation_notes, draw, table

quiet()
_ = torch.manual_seed(0)
SOURCES = {"executorch version": "executorch", "torchao test copy": "torchao"}

This project's `get_xnnpack_api(source)` and `get_xnnpack_quantizer(source, ...)` (`common/pt2e.py`, not torchao APIs) do just one thing: depending on `source`, import `XNNPACKQuantizer` and `get_symmetric_quantization_config` from one of the two paths above. Print the modules actually imported:

In [ ]:
for label, source in SOURCES.items():
    print(f"{label}: {get_xnnpack_api(source)[0].__module__}")

## A. On the Same CatPoolNet, Which Ops Each One Annotates

The middle of CatPoolNet is `relu`, `relu_1` → `cat` → `max_pool2d` → `conv2d_2`. On the same exported graph, call `quantizer.annotate` once for each (annotation only, no observers inserted, to make comparison easy), and draw the annotations under the nodes with `annotation_notes` (M03-2):

In [ ]:
def labeled(source, model):
    """export → call annotate only, return the annotated graph."""
    gm = export_module(model, example_inputs())
    get_xnnpack_quantizer(source, is_per_channel=True)[0].annotate(gm)
    return gm


model = CatPoolNet().eval()
graphs = {label: labeled(source, model) for label, source in SOURCES.items()}
MIDDLE = ["relu", "relu_1", "cat", "max_pool2d"]


def middle_notes(gm):
    notes = annotation_notes(gm, unlabeled=True)
    return {k: v for k, v in notes.items() if k in MIDDLE}

In [ ]:
draw(graphs["executorch version"], focus=MIDDLE, hops=0, notes=middle_notes(graphs["executorch version"]),
     title="executorch version: cat and max_pool2d are both annotated")

In [ ]:
draw(graphs["torchao test copy"], focus=MIDDLE, hops=0, notes=middle_notes(graphs["torchao test copy"]),
     title="torchao test copy: cat and max_pool2d are not annotated")

In both graphs relu has an output annotation; the difference is `cat` and `max_pool2d`. Compare the whole graph programmatically to confirm only these two differ:

In [ ]:
def has_label(gm):
    return {n.name: ANNOTATION_KEY in n.meta for n in gm.graph.nodes if n.op == "call_function"}


et, ao = (has_label(gm) for gm in graphs.values())
diff = sorted(n for n in et if et[n] != ao[n])
print("Nodes whose annotation differs between the two:", diff)
assert diff == ["cat", "max_pool2d"] and et["cat"] and not ao["cat"]

**`cat`**: the two cat annotators are written differently (read from the torchao 0.18 and executorch 1.5.0 source).

- The executorch version scans every node and directly matches `node.target == aten.cat.default`.
- The torchao test copy uses `get_source_partitions(gm.graph, [torch.cat])` to find the pattern. That function depends on `node.meta["source_fn_stack"]`, which `torch.export` in torch 2.14 no longer fills in, so it finds nothing, and does not report an error (M01-2).

**`max_pool2d`**: neither has an annotator of its own for max_pool2d. Its annotation comes from the last step, `propagate_annotation`: if the input is already annotated, max_pool2d shares qparams with the input (SharedQuantizationSpec, M03-4). The input `cat` is not annotated, so it is left unannotated too.

Use a model where max_pool2d comes straight after relu to verify that the torchao test copy's max_pool2d itself is not broken:

In [ ]:
pool_after_relu = nn.Sequential(nn.Conv2d(3, 8, 3), nn.ReLU(), nn.MaxPool2d(2)).eval()
pool_labeled = {}
for label, source in SOURCES.items():
    gm = labeled(source, pool_after_relu)
    pool_labeled[label] = has_label(gm)["max_pool2d"]
print("conv → relu → max_pool2d, is max_pool2d annotated:", pool_labeled)
assert all(pool_labeled.values())

Both annotate it: the max_pool2d in CatPoolNet is unannotated only because the upstream cat is unannotated.

## B. The executorch Version's Quantization Specs: static and dynamic

The executorch version's `get_symmetric_quantization_config(is_per_channel=True, is_dynamic=...)` produces a `QuantizationConfig` holding the qspecs for four positions. The two modes side by side:

In [ ]:
_, config_fn, _ = get_xnnpack_api("executorch")
configs = {"static (default)": config_fn(is_per_channel=True), "dynamic": config_fn(is_per_channel=True, is_dynamic=True)}
fields = ["input_activation", "output_activation", "weight", "bias"]
table([[f] + [qspec_summary(getattr(c, f)) for c in configs.values()] for f in fields], ["position"] + list(configs))

- weight is int8 per-channel symmetric in both modes, range [-127, 127], zero point fixed at 0.
- bias is `None` in both: on the Q/DQ graph the bias stays fp32. XNNPACK quantizes it to int32 during lowering (M08 overview).
- static: the activation scale is decided during calibration (this config uses `HistogramObserver`) and baked into the graph.
- dynamic: the input activation uses only a `PlaceholderObserver` as a placeholder, and the output activation is `None` (not quantized). The scale is computed **at run time** from the current input.

Dynamic mode is mainly for linear (e.g. LLMs). Do PTQ on Linear → ReLU → Linear and look at how the ops change in each mode after convert:

In [ ]:
mlp = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 4)).eval()
x_mlp = (torch.randn(2, 16),)
quantizer_cls = get_xnnpack_api("executorch")[0]
converted = {}
for label, cfg in configs.items():
    prepared = prepare_pt2e(export_module(mlp, x_mlp), quantizer_cls().set_global(cfg))
    prepared(*x_mlp)
    converted[label] = convert_pt2e(prepared)
table(diff_ops(converted["static (default)"], converted["dynamic"], "static", "dynamic", tablefmt="html"))

dynamic has extra `choose_qparams` and `getitem`. Draw only the part feeding the first linear's input:

In [ ]:
def upto_first_linear(gm):
    """The first linear and all of its upstream nodes."""
    linear = next(n for n in gm.graph.nodes if short(n.target) == "linear")
    kept, todo = set(), [linear]
    while todo:
        n = todo.pop()
        if n.name not in kept:
            kept.add(n.name)
            todo.extend(n.all_input_nodes)
    return kept


draw(converted["static (default)"], focus=upto_first_linear(converted["static (default)"]), hops=0,
     title="static: the q/dq scale is a constant")

In [ ]:
dyn = converted["dynamic"]
choose = next(n.name for n in dyn.graph.nodes if short(n.target) == "choose_qparams")
draw(dyn, focus=upto_first_linear(dyn), hops=0, notes={choose: "reads the current input\ncomputes (scale, zp)"},
     title="dynamic: scale is computed at run time")

In [ ]:
n_choose = sum(short(n.target) == "choose_qparams" for n in converted["dynamic"].graph.nodes)
n_linear = sum(short(n.target) == "linear" for n in converted["dynamic"].graph.nodes)
print(f"dynamic choose_qparams: {n_choose}; linear: {n_linear}")
assert n_choose == n_linear

- dynamic inserts a `choose_qparams` before the input of every linear, and two `getitem`s pull out the scale and zero point and pass them to the quantize and dequantize that follow.
- static has one more q/dq pair after the last linear (output quantization); dynamic's output activation is `None`, so it has none.

## C. Does `set_module_name` Take Effect on conv

M03-7 B found that with the torchao test copy, `set_module_name("stem", cfg)` silently has no effect on conv. Is the executorch version the same? Set it once each for TinyConvNet's `stem` (conv) and `fc` (linear), leaving everything else unquantized:

In [ ]:
tiny = TinyConvNet().eval()
rows, hit = [], {}
for label, source in SOURCES.items():
    cls, cfg_fn, _ = get_xnnpack_api(source)
    for module in ["stem", "fc"]:
        gm = export_module(tiny, example_inputs())
        cls().set_module_name(module, cfg_fn(is_per_channel=True)).annotate(gm)
        names = [n.name for n in gm.graph.nodes if n.op == "call_function" and ANNOTATION_KEY in n.meta]
        hit[(label, module)] = bool(names)
        rows.append([label, module, ", ".join(names) or "(no annotations)"])
table(rows, ["quantizer", "set_module_name", "annotated nodes"])

In [ ]:
assert all(not hit[(label, "stem")] and hit[(label, "fc")] for label in SOURCES)

The two are the same: `stem`'s conv gets no annotation at all, `fc`'s linear does, and neither reports an error.

The results match because this piece of code is written the same way in both (read from `xnnpack_quantizer_utils.py` in torchao 0.18 and executorch 1.5.0): the executorch version's conv annotator additionally supports conv_transpose and dynamic, but the part below is unchanged, and the module-name filtering also uses torchao's `get_module_name_filter`. So this is one problem showing up in two copies, not two problems:

- The linear annotator passes only the linear node through the filter.
- The conv annotator requires **every** node in the pattern to pass the filter, and the pattern includes the weight's `get_attr`. A `get_attr` has no `nn_module_stack`, so the filter skips the whole conv (M03-7 B has the full trace).

The same setting takes effect on linear, not on conv, with no error message at all; this looks like a quantizer bug (inference; whether it was reported upstream and whether it will be fixed was not tracked in this tutorial). To quantize only one section of conv, write your own Quantizer (M03-6).

## Summary

- The two `XNNPACKQuantizer`s have the same specs: activation int8 per-tensor affine, weight int8 per-channel symmetric, bias not quantized.
- The difference is in the annotator implementations: the torchao test copy's cat relies on `source_fn_stack` to find the pattern, which stops working in torch 2.14; the max_pool2d after cat, which relies on `propagate_annotation`, is left unannotated along with it.
- `set_module_name` silently has no effect on conv: the two filters are written the same way, so both behave alike.
- The three missed annotations seen in this notebook (cat, max_pool2d, conv under `set_module_name`) all report no error, and switching to the other quantizer does not necessarily avoid them. So whichever one you use and whatever settings you change, check two layers yourself:
  - **Annotations**: after `quantizer.annotate` or `prepare_pt2e`, draw them with `draw` + `annotation_notes` (section A of this notebook, M03-2) and confirm every op you want quantized is annotated.
  - **Q/DQ nodes**: after `convert_pt2e`, confirm that these ops really have q/dq before and after them (M01-2). The Q/DQ graph is what gets handed to the backend.
- From here on M08 always uses the executorch version: it is in the same package as `XnnpackPartitioner`, and the `.pte` export flow uses this one.